# Explorador interactivo de impacto raqueta-bola

Este cuaderno usa `table_tennis_simulation.py` para variar parámetros del golpe con sliders, simular la trayectoria resultante e identificar los momentos 1-6 de entrenamiento. También puede generar una animación embebida con la mesa, la bola antes/después del impacto y el gesto de raqueta ajustado con curvas de Bézier.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display
from ipywidgets import Button, FloatSlider, Output, VBox, interact
from matplotlib import animation

from table_tennis_simulation import (
    BALL_RADIUS, TABLE_HEIGHT, TABLE_LENGTH, TABLE_WIDTH,
    RacketImpactParameters, draw_racket, draw_table, identify_trajectory_moments,
    simulate_racket_impact,
)


In [ ]:
last_state = {}


def unit_vector(vector, fallback=(1.0, 0.0, 0.0)):
    vector = np.asarray(vector, dtype=float)
    norm = np.linalg.norm(vector)
    if norm < 1e-9:
        return np.asarray(fallback, dtype=float)
    return vector / norm


def bezier(points, u):
    points = np.asarray(points, dtype=float)
    if len(points) == 1:
        return points[0]
    return bezier((1 - u) * points[:-1] + u * points[1:], u)


def cubic_bezier_segment(p0, p3, v0, v3, duration, samples):
    p0 = np.asarray(p0, dtype=float)
    p3 = np.asarray(p3, dtype=float)
    p1 = p0 + np.asarray(v0, dtype=float) * duration / 3.0
    p2 = p3 - np.asarray(v3, dtype=float) * duration / 3.0
    return np.array([bezier([p0, p1, p2, p3], u) for u in np.linspace(0, 1, samples)])


def racket_gesture_path(contact_point, racket_velocity, samples=40):
    direction = unit_vector(racket_velocity)
    start = np.asarray(contact_point, dtype=float) - 100.0 * direction
    end = np.asarray(contact_point, dtype=float) + 100.0 * direction
    contact_velocity = np.asarray(racket_velocity, dtype=float)
    segment_duration = 0.08
    before = cubic_bezier_segment(start, contact_point, (0, 0, 0), contact_velocity, segment_duration, samples // 2)
    after = cubic_bezier_segment(contact_point, end, contact_velocity, (0, 0, 0), segment_duration, samples - samples // 2)
    return np.vstack([before[:-1], after])


def pre_impact_ball_path(contact_point, ball_velocity, samples=24, duration=0.12):
    start = np.asarray(contact_point, dtype=float) - unit_vector(ball_velocity, fallback=(-1, 0, 0)) * np.linalg.norm(ball_velocity) * duration
    return np.linspace(start, np.asarray(contact_point, dtype=float), samples)


def trajectory_stop_index(result, close_bounce_window=0.08):
    table_level = TABLE_HEIGHT + BALL_RADIUS
    floor_hits = np.where(result.x[2] <= BALL_RADIUS)[0]
    stop_index = int(floor_hits[0]) if len(floor_hits) else result.x.shape[1] - 1
    bounces = np.where(np.isclose(result.x[2], table_level, atol=1e-6) & (result.v[2] > 0))[0]
    if len(bounces) >= 2:
        bounce_times = result.t[bounces]
        close_pairs = np.where(np.diff(bounce_times) <= close_bounce_window)[0]
        if len(close_pairs):
            stop_index = min(stop_index, int(bounces[close_pairs[0] + 1]))
    return stop_index


def draw_ball(ax, center, color='white'):
    u, v_ang = np.mgrid[0:2 * np.pi:18j, 0:np.pi:9j]
    x = BALL_RADIUS * np.cos(u) * np.sin(v_ang) + center[0]
    y = BALL_RADIUS * np.sin(u) * np.sin(v_ang) + center[1]
    z = BALL_RADIUS * np.cos(v_ang) + center[2]
    ax.plot_surface(x, y, z, color=color, edgecolor='none')


def set_scene_axes(ax):
    ax.set_xlim(-300, TABLE_LENGTH + 700)
    ax.set_ylim(-400, TABLE_WIDTH + 400)
    ax.set_zlim(0, 1900)
    ax.set_xlabel('X (mm)')
    ax.set_ylabel('Y (mm)')
    ax.set_zlabel('Z (mm)')
    ax.view_init(23.5, -45)


In [ ]:
def summarize_moments(moments):
    for number, moment in moments.items():
        interval = f", intervalo={moment.interval}, punto medio={np.round(moment.midpoint, 1)}" if moment.interval else ""
        print(f"Momento {number}: {moment.name}; t={moment.time:.3f}s; punto={np.round(moment.point, 1)}{interval}")


def build_params(ball_vx=-2000, ball_vy=0, ball_vz=-1000,
                 omega_x=0, omega_y=0, omega_z=471,
                 friction=0.60, restitution=0.85,
                 angle_x=0, angle_y=-30, angle_z=0,
                 racket_vx=2000, racket_vy=0, racket_vz=1000):
    return RacketImpactParameters(
        ball_velocity=(ball_vx, ball_vy, ball_vz),
        ball_omega=(omega_x, omega_y, omega_z),
        rubber_friction=friction,
        rubber_restitution=restitution,
        racket_angle=(angle_x, angle_y, angle_z),
        racket_velocity=(racket_vx, racket_vy, racket_vz),
    )


def explore(**kwargs):
    params = build_params(**kwargs)
    result = simulate_racket_impact(params, dt=0.005, t_max=4.0)
    moments = identify_trajectory_moments(result)
    racket_path = racket_gesture_path(params.ball_position, params.racket_velocity)
    incoming_ball = pre_impact_ball_path(params.ball_position, params.ball_velocity)

    fig = plt.figure(figsize=(13, 5))
    ax = fig.add_subplot(121, projection='3d')
    draw_table(ax)
    ax.plot(result.x[0], result.x[1], result.x[2], color='purple', label='trayectoria post-impacto')
    ax.plot(incoming_ball[:, 0], incoming_ball[:, 1], incoming_ball[:, 2], color='orange', linestyle='--', label='bola pre-impacto')
    ax.plot(racket_path[:, 0], racket_path[:, 1], racket_path[:, 2], color='black', linestyle=':', label='gesto Bézier')
    draw_racket(ax, center=params.ball_position, angle=params.racket_angle)
    draw_ball(ax, params.ball_position)
    set_scene_axes(ax)
    ax.set_title('Mesa, trayectoria, raqueta y gesto')
    ax.legend(loc='upper left')

    ax2 = fig.add_subplot(122)
    t = result.t
    ax2.plot(t, result.x[2], label='altura bola')
    ax2.axhline(TABLE_HEIGHT + BALL_RADIUS, color='tab:gray', linestyle='--', label='nivel mesa + radio')
    for number, moment in moments.items():
        ax2.scatter(moment.time, moment.point[2], label=f'M{number}')
    ax2.set_xlabel('Tiempo (s)')
    ax2.set_ylabel('Z (mm)')
    ax2.legend()
    plt.tight_layout()

    last_state.update(params=params, result=result, moments=moments, racket_path=racket_path, incoming_ball=incoming_ball)
    summarize_moments(moments)
    return result, moments


In [ ]:
def make_impact_animation(state=None):
    state = last_state if state is None else state
    params = state['params']
    result = state['result']
    racket_path = state['racket_path']
    incoming_ball = state['incoming_ball']

    stop_index = trajectory_stop_index(result)
    frame_stride = max(1, round(0.02 / (result.t[1] - result.t[0])))
    post_indices = np.arange(0, stop_index + 1, frame_stride)
    if post_indices[-1] != stop_index:
        post_indices = np.append(post_indices, stop_index)
    pre_duration_ms = 120.0
    interval_ms = float((result.t[1] - result.t[0]) * frame_stride * 1000.0)
    total_frames = len(incoming_ball) + len(post_indices)

    fig = plt.figure(figsize=(8, 6))
    ax = fig.add_subplot(111, projection='3d')

    def update(frame):
        ax.clear()
        draw_table(ax)
        ax.plot(racket_path[:, 0], racket_path[:, 1], racket_path[:, 2], color='black', linestyle=':', alpha=0.65)
        if frame < len(incoming_ball):
            ball_center = incoming_ball[frame]
            racket_index = int(frame / max(1, len(incoming_ball) - 1) * (len(racket_path) // 2 - 1))
            post_index = 0
        else:
            post_index = int(post_indices[frame - len(incoming_ball)])
            ball_center = result.x[:, post_index]
            progress = (frame - len(incoming_ball)) / max(1, len(post_indices) - 1)
            racket_index = min(len(racket_path) // 2 + int(progress * (len(racket_path) // 2)), len(racket_path) - 1)
            ax.plot(result.x[0, :post_index + 1], result.x[1, :post_index + 1], result.x[2, :post_index + 1], color='purple')
        ax.plot(incoming_ball[:, 0], incoming_ball[:, 1], incoming_ball[:, 2], color='orange', linestyle='--', alpha=0.7)
        draw_racket(ax, center=racket_path[racket_index], angle=params.racket_angle)
        draw_ball(ax, ball_center)
        set_scene_axes(ax)
        elapsed = -pre_duration_ms / 1000 + frame * interval_ms / 1000
        ax.set_title(f'Animación embebida en tiempo real aproximado — t={elapsed:.2f}s')

    ani = animation.FuncAnimation(fig, update, frames=total_frames, interval=interval_ms, blit=False)
    plt.close(fig)
    return HTML(ani.to_jshtml())


In [ ]:
interact(
    explore,
    ball_vx=FloatSlider(value=-2000, min=-12000, max=12000, step=250, description='Bola Vx'),
    ball_vy=FloatSlider(value=0, min=-8000, max=8000, step=250, description='Bola Vy'),
    ball_vz=FloatSlider(value=-1000, min=-8000, max=8000, step=250, description='Bola Vz'),
    omega_x=FloatSlider(value=0, min=-1000, max=1000, step=25, description='Ωx'),
    omega_y=FloatSlider(value=0, min=-1000, max=1000, step=25, description='Ωy'),
    omega_z=FloatSlider(value=471, min=-1500, max=1500, step=25, description='Ωz'),
    friction=FloatSlider(value=0.60, min=0, max=1.5, step=0.05, description='Fricción'),
    restitution=FloatSlider(value=0.85, min=0, max=1.2, step=0.05, description='Restitución'),
    angle_x=FloatSlider(value=0, min=-90, max=90, step=2.5, description='Ángulo X'),
    angle_y=FloatSlider(value=-30, min=-90, max=90, step=2.5, description='Ángulo Y'),
    angle_z=FloatSlider(value=0, min=-180, max=180, step=5, description='Ángulo Z'),
    racket_vx=FloatSlider(value=2000, min=-8000, max=12000, step=250, description='Raqueta Vx'),
    racket_vy=FloatSlider(value=0, min=-8000, max=8000, step=250, description='Raqueta Vy'),
    racket_vz=FloatSlider(value=1000, min=-8000, max=8000, step=250, description='Raqueta Vz'),
);


In [ ]:
animation_output = Output()
animate_button = Button(description='Animar impacto', button_style='success')


def on_animate_clicked(_):
    with animation_output:
        animation_output.clear_output(wait=True)
        if not last_state:
            explore()
        display(make_impact_animation())


animate_button.on_click(on_animate_clicked)
display(VBox([animate_button, animation_output]))
